# 001 Create First Agent Without LLM

In this notebook we build a tiny agent using LangGraph.

Important: this first agent does not use OpenAI, Gemini, Groq, Ollama, or any LLM. It is only Python logic connected with LangGraph.

The output message will be: `Hello From Pakistan`.

## Very Simple Idea

Think about a small daily life process: making tea.

You can split it into steps:

1. Start
2. Boil water
3. Add tea
4. End

An agent is also like this. It follows steps. In LangGraph, those steps are called nodes, and the path between steps is called edges.

In this notebook our agent has only one real step: greet.

| Primitive | Purpose | Daily Life Example |
| --- | --- | --- |
| Node | A function that does one task and updates the state | One worker doing one job |
| Edge | A connection from one node to another | Road from one place to another |
| State | Data carried between nodes | Bag, memory box, or notebook |
| Graph Flow | The system that runs nodes in order | Traffic route or recipe steps |

## Step 1: Import Required Things

We import tools from Python and LangGraph.

`TypedDict` helps us define the shape of our state.

`Annotated` lets us attach extra behavior to a state field.

`StateGraph` is used to build the graph.

`START` and `END` are the beginning and ending points.

`add_messages` tells LangGraph how to add new messages into the message history.

In [3]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

ModuleNotFoundError: No module named 'langgraph'

## Step 2: Define The State

State means memory.

Imagine you are talking to a shopkeeper. The shopkeeper needs to remember what you said before answering. That remembered information is like state.

Here our state has one key: `messages`.

`messages` is a list. It can store chat messages.

`add_messages` means: when a node returns a new message, add it to the old messages instead of deleting everything.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

## Step 3: Initialize The Graph

Now we create an empty graph.

This is like opening a blank map before drawing roads on it.

`StateGraph(State)` means: build a graph that will carry the `State` we defined above.

In [ ]:
graph_builder = StateGraph(State)

## Step 4: Create The Greet Node

A node is just a Python function.

This function receives the current state.

Then it puts one message inside the state: `Hello From Pakistan`.

Daily life example: imagine a reception desk. A person comes in, and the receptionist does one job: greet the person. Our `greet` function is that receptionist.

In [ ]:
def greet(state: State) -> State:
    print("Greet state format: ", state)

    state["messages"] = ["Hello From Pakistan"]

    return {"messages": state["messages"]}

## Step 5: Add Node And Edges To The Graph

Now we place the `greet` function inside the graph.

`add_node("greet", greet)` means: create a node named `greet`, and when this node runs, call the `greet` function.

Then we add edges.

`START` to `greet` means the graph begins by running `greet`.

`greet` to `END` means after greeting, the graph is finished.

Daily life example: Start at your home, go to the shop, then come back. The roads are the edges.

In [ ]:
graph_builder.add_node("greet", greet)
graph_builder.add_edge(START, "greet")
graph_builder.add_edge("greet", END)

## Step 6: Compile The Graph

Compile means make the graph ready to run.

Before compiling, we are only designing the map.

After compiling, LangGraph can actually run it.

In [ ]:
graph = graph_builder.compile()

## Step 7: Display The Graph

Run this cell to see the graph object.

In many notebook editors, LangGraph can show a small flow diagram.

The flow is simple:

`START -> greet -> END`

In [ ]:
graph

## Step 8: Run The Graph

Now we call `graph.invoke()`.

Invoke means run the graph.

We give it an empty message list first: `{"messages": []}`.

The graph starts, goes to the greet node, puts `Hello From Pakistan` into messages, and then ends.

In [ ]:
graph.invoke({"messages": []})

## Step 9: Get Only The Message Text

The full output is a dictionary with message objects inside it.

If you want only the text, use this cell.

`["messages"]` gets the message list.

`[-1]` gets the latest message.

`.content` gets the real text inside the message.

In [ ]:
graph.invoke({"messages": ["Hi"]})["messages"][-1].content

## What Did We Build?

We built a very small agent without an LLM.

It does not think like ChatGPT. It does not call an API. It only follows rules that we wrote in Python.

But it is still useful for learning because real AI agents also use this same structure:

1. Keep state, like memory
2. Run a node, like one worker doing one job
3. Move through edges, like roads between tasks
4. Finish at END

Later, instead of writing `Hello From Pakistan` manually, we can put an LLM inside the node. Then the node can generate smart answers.

## Daily Life Example: Restaurant Agent

Imagine a small restaurant.

The customer says: I want biryani.

The restaurant process can be:

1. Take order
2. Send order to kitchen
3. Cook food
4. Serve food

Each step can be a node.

The order paper is the state because it carries information.

The path from waiter to kitchen to table is the graph flow.

This notebook has only one node, but the idea is the same.

## Restaurant Agent Code

Now we will build the restaurant example in code.

This is still without an LLM.

It is only Python functions connected with LangGraph.

The graph flow will be:

`START -> take_order -> send_to_kitchen -> cook_food -> serve_food -> END`

### Define Restaurant State

The state is like the order paper in the restaurant.

It carries the order and status from one step to the next step.

`messages` keeps a history of what happened.

In [ ]:
class RestaurantState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    order: str
    kitchen_status: str
    food_status: str
    served: bool

### Create The Restaurant Nodes

Each node is one worker doing one job.

`take_order` is like the waiter.

`send_to_kitchen` is like sending the order paper to the kitchen.

`cook_food` is like the chef.

`serve_food` is like the waiter bringing food back to the table.

In [ ]:
def take_order(state: RestaurantState) -> RestaurantState:
    order = state.get("order", "biryani")
    print("1. Taking order:", order)

    return {
        "order": order,
        "messages": [f"Waiter: Your order for {order} is noted."]
    }


def send_to_kitchen(state: RestaurantState) -> RestaurantState:
    order = state["order"]
    print("2. Sending order to kitchen:", order)

    return {
        "kitchen_status": "order sent to kitchen",
        "messages": [f"Kitchen received the order for {order}."]
    }


def cook_food(state: RestaurantState) -> RestaurantState:
    order = state["order"]
    print("3. Cooking food:", order)

    return {
        "food_status": f"{order} is ready",
        "messages": [f"Chef: The {order} is cooked and ready."]
    }


def serve_food(state: RestaurantState) -> RestaurantState:
    order = state["order"]
    print("4. Serving food:", order)

    return {
        "served": True,
        "messages": [f"Waiter: Here is your {order}. Enjoy your meal!"]
    }

### Build The Restaurant Graph

Now we connect the workers in the correct order.

These connections are called edges.

The graph tells LangGraph which step should run first, second, third, and fourth.

In [ ]:
restaurant_builder = StateGraph(RestaurantState)

restaurant_builder.add_node("take_order", take_order)
restaurant_builder.add_node("send_to_kitchen", send_to_kitchen)
restaurant_builder.add_node("cook_food", cook_food)
restaurant_builder.add_node("serve_food", serve_food)

restaurant_builder.add_edge(START, "take_order")
restaurant_builder.add_edge("take_order", "send_to_kitchen")
restaurant_builder.add_edge("send_to_kitchen", "cook_food")
restaurant_builder.add_edge("cook_food", "serve_food")
restaurant_builder.add_edge("serve_food", END)

restaurant_graph = restaurant_builder.compile()

### Display The Restaurant Graph

Run this cell to see the graph object or graph diagram in the notebook.

In [ ]:
restaurant_graph

### Run The Restaurant Agent

Now we start with an order: `biryani`.

The state moves through every node.

At the end, we get the complete final state.

In [ ]:
restaurant_result = restaurant_graph.invoke({
    "order": "biryani",
    "messages": []
})

restaurant_result

### Print The Message History

This shows the complete story of the restaurant agent.

Every node added one message.

In [ ]:
for message in restaurant_result["messages"]:
    print(message.content)

## Easy Explanation Of This Code

This restaurant agent is like a real restaurant workflow.

The customer order is the state.

The waiter, kitchen, chef, and waiter again are nodes.

The path between them is the edge.

LangGraph runs the workflow step by step.

No LLM is used here. The agent is not thinking. It is following fixed steps.

Later, we can replace one node with an LLM. For example, an LLM node could decide which kitchen station should receive the order.